# Notebook notes

- The noisy-input SNN uses the same global gain coefficient `65` and channel gains `(1, 1, 1, 3, 3, 1)` as the main whole-OT model.
- Noisy-input SNR uses the manuscript-aligned 15 s baseline window and 15-30 s stimulus window; in 10 ms display bins these are `slice(0, 1500)` and `slice(1500, 3000)`.
- A 2 s calcium-response lead is corrected on the input timeline before interpolation, so AUC quantification remains aligned to the nominal stimulus window.

In [ ]:
import brainpy as bp
import brainpy.math as bm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import jax
import os
import glob
from scipy import interpolate
from scipy.interpolate import interp1d


bm.enable_x64()
bm.set_platform('cpu')  # or 'gpu'

In [ ]:
acc = []

In [ ]:
# import pandas as pd
# import numpy as np

# # 读取CSV文件
# conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
# conn_matrix = conn_prob_df.values
# conn_matrix = conn_matrix.round(2)

# # 复制原始矩阵
# shuffled_matrix = conn_matrix.copy()

# # 获取需要洗牌的区域（第7行到最后，第7列到最后）
# rows_to_shuffle = shuffled_matrix[7:, 7:]

# # 将需要洗牌的区域展平、洗牌、然后重新reshape
# flattened = rows_to_shuffle.flatten()
# np.random.shuffle(flattened)  # 随机打乱
# shuffled_region = flattened.reshape(rows_to_shuffle.shape)

# # 将洗牌后的区域放回矩阵
# shuffled_matrix[7:, 7:] = shuffled_region

# # 创建新的DataFrame，保持原有的行名和列名
# shuffled_df = pd.DataFrame(shuffled_matrix, 
#                           index=conn_prob_df.index, 
#                           columns=conn_prob_df.columns)

# # 保存为Excel文件
# shuffled_df.to_csv('./neuron_connections_shuffle.csv')

# print("洗牌完成！文件已保存为 './shuffled_neuron_connections.csv'")

In [ ]:
##LOOMING PART
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)

# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NLooming0.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长30秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        # 返回监控变量
        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
##LOOMING PART
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)

# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NLooming001.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        # 返回监控变量
        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
##LOOMING PART
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)

# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NLooming005.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
##LOOMING PART
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)

# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NLooming01.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
##LOOMING PART
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)

# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NLooming02.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
##LOOMING PART
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)

# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NLooming05.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
# 单独损毁
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)


# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NSD0.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长30秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
# 单独损毁
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)


# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NSD001.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        # 返回监控变量
        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
# 单独损毁
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)


# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NSD005.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
# 单独损毁
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)


# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NSD01.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
# 单独损毁
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)


# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NSD02.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())

In [ ]:
# 单独损毁
# 单独损毁
neuron_num_df = pd.read_csv('./neuron_number.csv')
neuron_types = neuron_num_df['Type'].tolist()
neuron_numbers = neuron_num_df.set_index('Type')['number'].to_dict()


# 读取连接概率矩阵
conn_prob_df = pd.read_csv('./neuron_connections_whole.csv', index_col=0)
conn_matrix = conn_prob_df.values
conn_matrix = conn_matrix.round(2)


# 读取损毁顺序的CSV文件
# NLooming_order = pd.read_csv('./NoisyLooming_Order_test.csv')  # 假设文件名为damage_order.csv
# NLooming_shuffle_sequence = NLooming_order.iloc[:, 0].tolist()  # 提取第一列作为损毁顺序

# 简易反卷积函数 (优化版本)，将钙信号转化成脉冲序列再转化成电流信号输入到RGC中
def simple_deconvolution(calcium_signal, tau=0.5, dt=0.001):
    """
    基于钙动力学模型的简易反卷积
    :param calcium_signal: 钙信号序列
    :param tau: 钙衰减时间常数(秒)
    :param dt: 时间步长(秒)
    :return: 脉冲序列
    """
    n = len(calcium_signal)
    spikes = np.zeros(n)
    c_est = np.zeros(n)
    alpha = np.exp(-dt/tau)
    beta = dt/tau
    
    # 添加小常数防止除零
    beta = max(beta, 1e-6)
    
    for t in range(1, n):
        c_pred = c_est[t-1] * alpha
        s_t = max(0, (calcium_signal[t] - c_pred) / beta)
        c_est[t] = c_pred + s_t * beta
        spikes[t] = s_t
    
    return spikes

# 1. 读取钙信号数据
data = pd.read_excel('NSD05.xlsx', header=0, sheet_name='Sheet1')
time_points = data['Time'].values
calcium_signals = data.iloc[:, 1:7].values  # 6个RGC的钙信号

# 2. 计算数据时间分辨率
time_diffs = np.diff(time_points)
dt_data = np.median(time_diffs)  # 原始数据的时间分辨率(秒)
print(f"原始数据时间分辨率: {dt_data:.4f} 秒")

# 3. 反卷积参数 (斑马鱼RGC推荐值)
tau_calcium = 0.5  # 钙衰减时间常数(秒)

# 4. 反卷积为脉冲序列
spike_trains = np.zeros_like(calcium_signals)
for i in range(6):
    spike_trains[:, i] = simple_deconvolution(
        calcium_signals[:, i], 
        tau=tau_calcium,
        dt=dt_data
    )

# 5. 转换为发放率 (使用高斯滤波器替代滑动窗口)
from scipy.ndimage import gaussian_filter1d

# 计算高斯滤波器的sigma (对应~100ms的时间窗)
sigma = 0.1 / dt_data  # 100ms窗口对应的sigma值

# 确保sigma至少为1
sigma = max(sigma, 1.0)

firing_rates = np.zeros_like(spike_trains)
for i in range(6):
    firing_rates[:, i] = gaussian_filter1d(spike_trains[:, i], sigma=sigma)

# 6. 转换为输入电流 (增益调整)
current_gain = 1  # pA/(spikes/s)
rgc_currents = firing_rates * current_gain

# 7. 创建模拟时间序列 (转换为毫秒)
dt_sim = 0.1  # 模拟时间步长(ms)
sim_duration = 30000  # 模拟时长25秒(ms)
sim_times = np.arange(0, sim_duration, dt_sim)  # 毫秒单位

# 8. 插值到模拟时间点
rgc_inputs = np.zeros((len(sim_times), 6))

for i in range(6):
    # 原始数据时间点 (转换为毫秒)
    # Align calcium responses to the manuscript's nominal stimulus timeline.
    # The raw calcium response leads the nominal stimulus epoch by 2 s.
    calcium_response_lead_ms = 2000
    data_times_ms = time_points * 1000 + calcium_response_lead_ms
    
    # 创建插值函数 (使用线性插值)
    f = interp1d(
        data_times_ms, 
        rgc_currents[:, i],
        kind='linear',
        bounds_error=False,
        fill_value=(rgc_currents[0, i], rgc_currents[-1, i])
    )
    
    # 插值到模拟时间点
    rgc_inputs[:, i] = f(sim_times)

# 9. 转换为JIT兼容的数组
# Input scaling used in the final whole-OT model.
input_gain_coeff = 65
rgc_inputs_bm = bm.array(rgc_inputs)  # shape (6, simulation steps)
input_channel_gains = (1, 1, 1, 3, 3, 1)  # RGC_SO, RGC_S12, RGC_S34, RGC_S56, RGC_SGC, RGC_SAC
inp1 = rgc_inputs_bm[:, 0] * input_channel_gains[0] * input_gain_coeff
inp2 = rgc_inputs_bm[:, 1] * input_channel_gains[1] * input_gain_coeff
inp3 = rgc_inputs_bm[:, 2] * input_channel_gains[2] * input_gain_coeff
inp4 = rgc_inputs_bm[:, 3] * input_channel_gains[3] * input_gain_coeff
inp5 = rgc_inputs_bm[:, 4] * input_channel_gains[4] * input_gain_coeff
inp6 = rgc_inputs_bm[:, 5] * input_channel_gains[5] * input_gain_coeff

class Exponential(bp.Projection):
    def __init__(self, pre, post, prob, g_max, tau):
        super().__init__()
        self.proj = bp.dyn.ProjAlignPostMg2(
            pre=pre,
            delay=None, 
            comm=bp.dnn.EventCSRLinear(bp.conn.FixedProb(prob, pre=pre.num, post=post.num), g_max),
            syn=bp.dyn.Expon.desc(post.num, tau=tau),
            out=bp.dyn.CUBA.desc(),
            post=post,
        )

## 正式网络
class Whole_OT(bp.DynSysGroup):
    def __init__(self, method='exp_auto'):
        super().__init__()

        # 神经元参数
        pars = dict(V_rest=-60., V_th=-50., V_reset=-60., tau=20., tau_ref=5., R=1.,
                    V_initializer=bp.init.Normal(-55., 2.), method=method)

        # 存储神经元和突触的字典
        self.neurons = {}
        self.synapses = {}
        
        # 首先创建所有神经元组
        for ntype in neuron_types:
            num = int(neuron_numbers[ntype])
            # 创建神经元组并直接设置为属性
            neuron_group = bp.dyn.LifRef(num, **pars)
            setattr(self, ntype, neuron_group)
            self.neurons[ntype] = neuron_group

        print("神经元构建完成")
        
        # 然后创建所有突触连接
        for i, src_type in enumerate(neuron_types):
            for j, tar_type in enumerate(neuron_types):
                prob = conn_matrix[i, j]
                
                # 跳过零概率连接
                if prob <= 0:
                    continue
                
                # 确定突触参数
                g_max = 0.6  # 默认权重
                E = 0.0      # 默认兴奋性反转电位
                tau = 5.
                
                # 如果是抑制性神经元
                if src_type.startswith('I') or 'I_' in src_type:
                    tau = 10.
                    g_max = -6.7
                
                # 创建唯一的突触名称
                syn_name = f"{src_type}_to_{tar_type}"
                
                # 创建突触连接并设置为属性
                synapse = Exponential(
                    pre=self.neurons[src_type],
                    post=self.neurons[tar_type],
                    prob=prob,
                    g_max=g_max,
                    tau=tau
                )
                setattr(self, syn_name, synapse)
                self.synapses[syn_name] = synapse
                
                # print(f"Created synapse: {syn_name} (prob={prob}, g_max={g_max})")

        print("突触构建完成")

    def update(self, inp1, inp2, inp3, inp4, inp5, inp6):
        # 更新输入神经元
        self.RGC_SO(inp1)
        self.RGC_S12(inp2)
        self.RGC_S34(inp3)
        self.RGC_S56(inp4)
        self.RGC_SGC(inp5)
        self.RGC_SAC(inp6)

        # 更新所有神经元（包括非输入神经元）
        for ntype in neuron_types:
            # 跳过已经更新过的输入神经元
            if ntype not in ['RGC_SO', 'RGC_S12', 'RGC_S34', 'RGC_S56', 'RGC_SGC', 'RGC_SAC']:
                getattr(self, ntype)()

        # 更新所有突触
        for synapse in self.synapses.values():
            synapse()

        return (
            self.TPN_O.spike, 
            self.TPN_E.spike
        )


# 确保在创建网络前定义了必要的全局变量
# neuron_types, neuron_numbers, conn_matrix 需要预先定义
net_whole_OT = Whole_OT()
    
def run_net(t, inp1, inp2, inp3, inp4, inp5, inp6):
    """更新函数: t是当前时间(毫秒), _是占位符"""
    bp.share.save(t=t)    
    # 更新网络并返回监控值
    return net_whole_OT(inp1, inp2, inp3, inp4, inp5, inp6)


with bm.environment(dt=dt_sim):
    
    step_indices = bm.arange(len(sim_times))
    #tpn_o_spike, tpn_o_v, tpn_e_spike, tpn_e_v, rgc_v, rgc_spike = bm.for_loop(run_net, (times, net_inp1, net_inp2, net_inp3, net_inp4, net_inp5, net_inp6))
    outputs = bm.for_loop(
        run_net, 
        (step_indices, inp1, inp2, inp3, inp4, inp5, inp6), 
    )
    # 解包结果
    tpn_o_spike, tpn_e_spike = outputs 

rate1 = bp.measure.firing_rate(tpn_e_spike, width=1000.)
rate2 = bp.measure.firing_rate(tpn_o_spike, width=1000.)

# Manuscript-aligned AUC windows.
# Display-bin slices make the intended windows explicit to readers: 0-15 s and 15-30 s.
display_bin_ms = 10
display_bin_steps = int(display_bin_ms / dt_sim)
baseline_window_nominal_slice = slice(0, 1500)
signal_window_nominal_slice = slice(1500, 3000)
baseline_slice = slice(
    baseline_window_nominal_slice.start * display_bin_steps,
    baseline_window_nominal_slice.stop * display_bin_steps,
)
signal_slice = slice(
    signal_window_nominal_slice.start * display_bin_steps,
    signal_window_nominal_slice.stop * display_bin_steps,
)

print(rate1[signal_slice].sum())
print(rate1[baseline_slice].sum())
print(rate1[signal_slice].sum()/rate1[baseline_slice].sum())

print(rate2[signal_slice].sum())
print(rate2[baseline_slice].sum())
print(rate2[signal_slice].sum()/rate2[baseline_slice].sum())